# 00 Stationarity and Unit Roots

Before you regress one macro series on another, ask the boring question first: **are these series even stationary?** If not, the OLS coefficient table is going to lie to you in ways HAC SE cannot fix.


## Table of Contents
- [Stationarity, intuitively](#stationarity-intuitively)
- [The spurious regression trap](#spurious)
- [ADF and KPSS tests](#adf-kpss)
- [Differencing strategy](#differencing)
- [Cointegration in one paragraph](#cointegration)
- [Apply to the macro panel](#macro-panel)
- [Checkpoint (Self-Check)](#checkpoint-self-check)
- [Solutions (Reference)](#solutions-reference)


## Why This Notebook Matters
Macro series like GDP, CPI, and unemployment levels are almost always non-stationary. If you regress one on another in levels, the t-stats and R² in the output table can be wildly misleading — Granger and Newbold (1974) showed you can get a 'significant' relationship between two unrelated random walks 75%+ of the time.

You will learn:
- what stationarity means (mean, variance, autocovariance constant in time),
- the **spurious regression** problem and why HAC SE does not save you,
- how ADF and KPSS tests work, and why running both is better than running one,
- the differencing decision: levels vs first differences vs log differences,
- a one-paragraph mention of cointegration so you know it exists.

## Prerequisites (Quick Self-Check)
- Sections 00a (statistics primer) and 00b (foundations).
- Section 01 (data) — you have a quarterly macro panel handy.
- Comfort with pandas time-series basics (`shift`, `diff`, `pct_change`).

## What You Will Produce
- A stationarity verdict table for each series in the macro panel.
- A clear opinion about which transformation to use before regressing in §02.

## Common Pitfalls
- Treating ADF as the final word — it has low power on small samples.
- Differencing reflexively without checking. Over-differencing destroys signal.
- Confusing stationarity with 'looks flat' — a series with a slow trend can still pass an eye test.
- Forgetting that growth rates and log-differences are usually the right unit for macro work.

## Quick Fixes (When You Get Stuck)
- If `adf_test` errors on too few observations, drop NaNs first.
- If KPSS returns a p-value pinned at the boundary (0.01 or 0.10), accept it — those are the tabulated extremes.
- If ADF and KPSS disagree, that is a real signal of trend-stationarity or borderline integration. Read the verdict explanation in the guide.

## Matching Guide
- `docs/guides/01b_time_series_diagnostics/00_stationarity_and_unit_roots.md`


## How To Use This Notebook
- Work section-by-section; don't skip the markdown.
- Most code cells are incomplete on purpose: replace TODOs and `...`, then run.
- After each section, write 2–4 sentences answering the interpretation prompts (what changed, why it matters).
- Use the **Checkpoint (Self-Check)** section to catch mistakes early.
- Use **Solutions (Reference)** only to unblock yourself; then re-implement without looking.
- Use the matching guide for the math and a slightly deeper treatment.


<a id="environment-bootstrap"></a>
## Environment Bootstrap
Run this cell first. It makes the repo importable and defines common directories.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    p = start
    for _ in range(8):
        if (p / 'src').exists() and (p / 'docs').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root. Start Jupyter from the repo root.')


PROJECT_ROOT = find_repo_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
SAMPLE_DIR = DATA_DIR / 'sample'

PROJECT_ROOT


<a id="stationarity-intuitively"></a>
## Stationarity, intuitively

A series is **(weakly) stationary** if three things stay constant over time:
1. its **mean**,
2. its **variance**,
3. its **autocovariance at each lag** ($\mathrm{Cov}(y_t, y_{t-k})$ depends only on $k$, not on $t$).

Plain English: the series wanders around a fixed center with a fixed amount of jitter, and how strongly today depends on yesterday is the same in 1995 as in 2020.

Three useful caricatures:
- **White noise** — every observation is an independent draw. Stationary.
- **Random walk** — $y_t = y_{t-1} + \varepsilon_t$. Variance grows with $t$. Non-stationary (a unit root).
- **Random walk with drift** — same plus a constant. Both mean and variance grow with $t$.

Most macro **levels** (real GDP, CPI, employment) behave like the third one. Most macro **growth rates** (GDP growth, inflation, change in unemployment) behave much closer to the first.


### Your Turn (1): Eyeball three simulated processes


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
T = 400

# TODO: build three series of length T:
#   noise: N(0, 1) draws
#   walk: cumulative sum of N(0, 1) draws
#   walk_drift: cumulative sum of N(0.05, 1) draws
noise = ...
walk = ...
walk_drift = ...

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, s, title in zip(axes, [noise, walk, walk_drift], ['White noise', 'Random walk', 'Walk + drift']):
    ax.plot(s)
    ax.set_title(title)
fig.tight_layout()


<a id="spurious"></a>
## The spurious regression trap

Two **independent** random walks, regressed on each other, will routinely produce a tiny p-value. The OLS machinery has no way to know the data are non-stationary; it just minimizes squared residuals.

The simulation below is the canonical demo. Run it, look at how often you reject $H_0: \beta = 0$ at 5%, and notice that the answer is *not* 5%.


### Your Turn (1): Spurious-regression Monte Carlo


In [ ]:
import statsmodels.api as sm

rng = np.random.default_rng(42)
T, N = 200, 500

rejections_levels = 0
rejections_diffs = 0

for _ in range(N):
    y = rng.normal(size=T).cumsum()
    x = rng.normal(size=T).cumsum()

    # Levels regression
    Xc = sm.add_constant(x)
    res = sm.OLS(y, Xc).fit()
    if res.pvalues[1] < 0.05:
        rejections_levels += 1

    # First-differences regression
    dy = np.diff(y)
    dx = np.diff(x)
    Xc_d = sm.add_constant(dx)
    res_d = sm.OLS(dy, Xc_d).fit()
    if res_d.pvalues[1] < 0.05:
        rejections_diffs += 1

print(f'Levels:           rejected H0 in {rejections_levels}/{N} ({100*rejections_levels/N:.1f}%)')
print(f'First diffs:      rejected H0 in {rejections_diffs}/{N} ({100*rejections_diffs/N:.1f}%)')
print('Expected under H0 with valid SE: ~5%.')


### Your Turn (2): Interpretation prompt

Write 4–6 sentences answering:
- Why is the levels rejection rate so much larger than 5%?
- Why does first-differencing fix it? (Hint: the differences of two independent walks are independent white noise.)
- HAC SE corrects for autocorrelation in residuals. Why doesn't HAC fix the spurious-regression problem?


In [ ]:
notes = """
...
"""
print(notes)


<a id="adf-kpss"></a>
## ADF and KPSS tests

The two main unit-root tests have **opposite null hypotheses** on purpose:

| Test | $H_0$ | $H_1$ | Reject (small p) means |
|---|---|---|---|
| **ADF** | series has a unit root | series is stationary | evidence of stationarity |
| **KPSS** | series is stationary | series has a unit root | evidence of non-stationarity |

Run both. The four cases:
- **ADF rejects, KPSS does not** → call it stationary.
- **ADF does not reject, KPSS rejects** → call it non-stationary (likely unit root).
- **Both reject** → likely *trend-stationary* or borderline. Try removing a linear trend, or use ADF with a trend term.
- **Neither rejects** → low power. Try more data, a larger sample, or accept that you cannot tell.

The repo has a small wrapper that runs both and tags the verdict.


### Your Turn (1): Run ADF and KPSS on the three caricatures


In [ ]:
from src import econometrics

rng = np.random.default_rng(7)
T = 400
noise = pd.Series(rng.normal(size=T))
walk = pd.Series(rng.normal(size=T).cumsum())
trend_stat = pd.Series(0.02 * np.arange(T) + rng.normal(size=T))  # linear trend + noise

df_caricatures = pd.DataFrame({'noise': noise, 'walk': walk, 'trend_stationary': trend_stat})

# TODO: call econometrics.stationarity_table on the three series.
table = ...
table


### Your Turn (2): Re-test the trend-stationary series with a trend term

When a series wanders around a deterministic trend (constant slope plus noise), the constant-only ADF spec is misspecified. Use `regression='ct'` to allow a trend.


In [ ]:
# TODO: call econometrics.adf_test with regression='ct' on trend_stat.
res_ct = ...
res_ct.as_dict()


<a id="differencing"></a>
## Differencing strategy

The order of integration $d$ is the number of differences needed to make a series stationary:
- $I(0)$: stationary in levels.
- $I(1)$: stationary in first differences. Most macro levels live here (GDP, CPI, employment).
- $I(2)$: stationary in second differences. Rare; sometimes price levels under high inflation.

**Practical recipe** for macro work:
1. For *levels* of stocks, prices, GDP, CPI: take **log differences** (≈ percentage growth, well-behaved).
2. For *rates* (unemployment, fed funds, yields): take **first differences** (already in percent, so logs make less sense).
3. For *growth rates* you computed elsewhere: usually already $I(0)$, leave them alone.

Over-differencing introduces an MA(1) error structure with a unit root — the cure becomes the disease. Difference once, retest, only difference again if the first differences still fail.


### Your Turn (1): Show that first-differencing a random walk gives white noise


In [ ]:
rng = np.random.default_rng(11)
walk = pd.Series(rng.normal(size=400).cumsum())

# TODO: build dwalk = first difference of walk; drop the first NaN.
# Then run stationarity_table on a DataFrame with 'walk' and 'dwalk'.
...


<a id="cointegration"></a>
## Cointegration in one paragraph

Two non-stationary series can have a stationary linear combination. That is **cointegration**, and it is the rigorous version of the intuition 'these series move together over the long run.'

Classic examples: short and long interest rates, log consumption and log income, the spot and futures prices of the same commodity. If $y$ and $x$ are both $I(1)$ but $y - \beta x$ is $I(0)$, then the levels regression $y = \alpha + \beta x + u$ is **not** spurious — it is recovering a real long-run equilibrium, and the residuals being stationary is the proof.

The Engle–Granger two-step test is exactly that: regress in levels, then run ADF on the residuals. If the residuals reject the unit-root null, you have cointegration. Modern work uses Johansen's test for multiple variables, but the intuition is the same.

For this curriculum we **do not chase cointegration explicitly**. The macro regressions in §02 use growth-rate targets, which sidesteps it. But you should recognize the term when you see it in macro papers.


<a id="macro-panel"></a>
## Apply to the macro panel

Now run stationarity diagnostics on the actual quarterly macro panel you built in §01. The point is to make a defensible decision about which transformation to feed into §02 regression.


### Your Turn (1): Load the panel


In [ ]:
path = PROCESSED_DIR / 'macro_quarterly.csv'
if path.exists():
    df = pd.read_csv(path, index_col=0, parse_dates=True)
else:
    df = pd.read_csv(SAMPLE_DIR / 'macro_quarterly_sample.csv', index_col=0, parse_dates=True)

df.tail()


### Your Turn (2): Stationarity table on raw levels

Pick the level columns: `GDPC1`, `CPIAUCSL`, `INDPRO`, `RSAFS`, `UNRATE`, `FEDFUNDS`, `T10Y2Y`. The first four are likely $I(1)$; the last three are rates and ambiguous.


In [ ]:
level_cols = ['GDPC1', 'CPIAUCSL', 'INDPRO', 'RSAFS', 'UNRATE', 'FEDFUNDS', 'T10Y2Y']
level_cols = [c for c in level_cols if c in df.columns]

# TODO: run stationarity_table on df[level_cols].
levels_table = ...
levels_table


### Your Turn (3): Build transformed columns and re-test

- For `GDPC1`, `CPIAUCSL`, `INDPRO`, `RSAFS` (positive-valued levels): take 100·log diff.
- For `UNRATE`, `FEDFUNDS` (rates): take first differences.
- `T10Y2Y` is already a spread, often borderline stationary.


In [ ]:
trans = pd.DataFrame(index=df.index)
for col in ['GDPC1', 'CPIAUCSL', 'INDPRO', 'RSAFS']:
    if col in df.columns:
        trans[f'{col}_logdiff'] = 100.0 * np.log(df[col]).diff()
for col in ['UNRATE', 'FEDFUNDS']:
    if col in df.columns:
        trans[f'{col}_diff'] = df[col].diff()
if 'T10Y2Y' in df.columns:
    trans['T10Y2Y'] = df['T10Y2Y']

# TODO: drop NaN rows, then run stationarity_table on the transformed DataFrame.
transformed_table = ...
transformed_table


### Your Turn (4): Decide and document

Write 6–10 sentences:
- Which series in levels failed the stationarity test? Did you expect that?
- After transformation, which series cleared the table? Any holdouts?
- Which transformations will you use as your default in §02 regression?
- What is one risk you are still carrying after this analysis?


In [ ]:
decision_notes = """
...
"""
print(decision_notes)


<a id="checkpoint-self-check"></a>
## Checkpoint (Self-Check)
Run a few asserts and write 2–3 sentences summarizing what you verified.


In [ ]:
# Sanity checks (uncomment after you build the tables above):
# assert 'verdict' in levels_table.columns
# assert (levels_table['verdict'] == 'non-stationary (unit root)').sum() >= 2
# assert (transformed_table['verdict'] == 'stationary').sum() >= 3
...


## Extensions (Optional)
- Repeat the spurious-regression Monte Carlo with HAC SE (`maxlags=4`). Does the rejection rate drop to 5%? (Spoiler: no — but report what you see.)
- Implement the Engle–Granger cointegration test on `GDPC1` and `INDPRO` levels: regress one on the other, then ADF the residuals. Discuss what you find.
- Use rolling 10-year windows to compute the ADF statistic on `UNRATE` and plot it. Does the verdict change over time?


## Reflection
- Before this notebook, would you have regressed `GDPC1` on `CPIAUCSL` levels and trusted the t-statistics? Why or why not?
- What is the cheapest ongoing habit you can adopt to avoid spurious regressions in your day-to-day analysis?


<a id="solutions-reference"></a>
## Solutions (Reference)

Try the TODOs first. Use these only to unblock yourself or to compare approaches.

<details><summary>Solution: Three caricatures</summary>

```python
noise = rng.normal(size=T)
walk = rng.normal(size=T).cumsum()
walk_drift = (0.05 + rng.normal(size=T)).cumsum()
```

</details>

<details><summary>Solution: Stationarity table on the three caricatures</summary>

```python
from src import econometrics
table = econometrics.stationarity_table(df_caricatures, df_caricatures.columns)
```

Expected verdicts: `noise` → stationary, `walk` → non-stationary, `trend_stationary` → conflicting under `regression='c'`. Re-run ADF with `regression='ct'` for the trend-stationary one.

</details>

<details><summary>Solution: ADF with trend term</summary>

```python
res_ct = econometrics.adf_test(trend_stat, regression='ct')
```

</details>

<details><summary>Solution: First-difference a random walk</summary>

```python
dwalk = walk.diff().dropna()
tbl = econometrics.stationarity_table(
    pd.DataFrame({'walk': walk, 'dwalk': dwalk.reindex(walk.index)}).dropna(),
    ['walk', 'dwalk'],
)
```

</details>

<details><summary>Solution: Macro panel — levels and transformed</summary>

```python
levels_table = econometrics.stationarity_table(df[level_cols].dropna(), level_cols)
transformed_table = econometrics.stationarity_table(trans.dropna(), trans.columns)
```

On the bundled sample with ~160 quarters you should see GDPC1, CPIAUCSL, INDPRO, RSAFS clearly non-stationary in levels and stationary after log-differencing. UNRATE and FEDFUNDS are often borderline in levels and clearly stationary after first-differencing. T10Y2Y is usually already stationary.

</details>
